In [1]:
import sys, time
from pprint import pprint
# time.sleep(10)

pprint(sys.version)
from pathlib import Path

HOME_DIR = Path.home()
CODE_DIR = HOME_DIR / 'synthesizrr' / 'src'
sys.path.insert(-2, '/home/ec2-user/anaconda3/envs/hft4/lib/python3.11/site-packages')
sys.path.insert(-2, '/home/ec2-user/anaconda3/envs/hft4/bin/')
sys.path.insert(-2, str(CODE_DIR))
pprint(sys.path)

import synthergent
from typing import *
import time, glob, os, sys, boto3, numpy as np, pandas as pd, json, requests, gc
from pandas.core.frame import DataFrame as PandasDataFrame, Series as PandasSeries
from pathlib import Path

RAY_TMP_DIR = '/tmp/ray/'

from synthergent.base.util import *
from synthergent.base.data import *
from synthergent.base.constants import *
from synthergent.base.framework import *
from synthergent.base.data.reader import Reader
from synthergent.base.data.writer import Writer
from synthergent.base.framework.dl.torch import *
from synthergent.base.framework.task_data import DataSplit, Datasets, TaskData
import synthergent.base.algorithm
import synthergent.base.metric
from synthergent.base.framework.task.classification import _normalize_label
import datasets as ds
# from datasets import load_dataset, load_from_disk
from synthergent.base.framework.trainer.RayTuneTrainer import _ray_metric_str
from termcolor import COLORS as TERMCOLOR_COLORS
from termcolor import colored
import ray
from ray.util.dask import ray_dask_get, enable_dask_on_ray, disable_dask_on_ray

from pprint import pprint

os.environ['CUDA_VISIBLE_DEVICES'] = '4,6'

## print = Tracker.default().info

# import hvplot.pandas
# import holoviews as hv
# import plotly.express as px
# import plotly.io as pio
# from IPython.display import display
# from bokeh.palettes import Spectral, Set2, Set3

# pio.templates.default = 'plotly_white'
# hvplot.extension('plotly')
# import numpy as np
# import pandas as pd
# import plotly.graph_objects as go
# import plotly.io as pio
# # pio.renderers.default='iframe'
# # hvplot.extension('bokeh')

# from bokeh.resources import INLINE as BOKEH_INLINE
# from bokeh.io import output_notebook as bokeh_output_notebook
# bokeh_output_notebook(BOKEH_INLINE)


from synthergent import Synthergent, Cleaner
import synthergent.cleaner

'3.11.8 | packaged by conda-forge | (main, Feb 16 2024, 20:53:32) [GCC 12.3.0]'
['/opt/conda/envs/hft4/lib/python311.zip',
 '/opt/conda/envs/hft4/lib/python3.11',
 '/opt/conda/envs/hft4/lib/python3.11/lib-dynload',
 '/home/ec2-user/anaconda3/envs/hft4/lib/python3.11/site-packages',
 '/home/ec2-user/anaconda3/envs/hft4/bin/',
 '/efs/litmus-server/users/adivekar/synthesizrr/src',
 '',
 '/opt/conda/envs/hft4/lib/python3.11/site-packages']


In [2]:
from datasets import load_dataset as hf_load_dataset
cosmopedia = hf_load_dataset("HuggingFaceTB/cosmopedia", "stories", split="train", num_proc=12)

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/43 [00:00<?, ?it/s]

HfHubHTTPError: 503 Server Error: Service Temporarily Unavailable for url: https://huggingface.co/api/datasets/HuggingFaceTB/cosmopedia/revision/0ae6ec63f91742bd2d1eaef4f02232c55d719385

In [2]:
import ray, dask
from ray.util.dask import enable_dask_on_ray
ray.shutdown()
pprint(ray.init(
    address='ray://10.0.152.54:10001',
    ignore_reinit_error=True,
    _temp_dir=str(RAY_TMP_DIR),
    runtime_env={"py_modules": [
        synthergent,
    ]},
))
enable_dask_on_ray()
pprint(ray.cluster_resources())

2024-12-15 17:54:27,010	INFO client_builder.py:243 -- Passing the following kwargs to ray.init() on the server: ignore_reinit_error
I0000 00:00:1734285267.026863    7856 config.cc:230] gRPC experiments enabled: call_status_override_on_cancellation, event_engine_dns, event_engine_listener, http2_stats_fix, monitoring_experiment, pick_first_new, trace_record_callops, work_serializer_clears_time_cache
2024-12-15 17:54:27,554	INFO packaging.py:530 -- Creating a file package for local directory '/efs/litmus-server/users/adivekar/synthesizrr/src/synthergent'.
2024-12-15 17:54:27,914	INFO packaging.py:358 -- Pushing file package 'gcs://_ray_pkg_7dead4843a4020ae.zip' (4.30MiB) to Ray cluster...
2024-12-15 17:54:27,968	INFO packaging.py:371 -- Successfully pushed file package 'gcs://_ray_pkg_7dead4843a4020ae.zip'.
SIGTERM handler is not set because current thread is not the main thread.


ClientContext(dashboard_url='127.0.0.1:8265',
              python_version='3.11.8',
              ray_version='2.9.2',
              ray_commit='fce7a361807580953364e2da964f9498f3123bf9',
              protocol_version='2023-06-27',
              _num_clients=1,
              _context_to_restore=<ray.util.client._ClientContext object at 0x7f6ff2d96390>)
{'CPU': 864.0,
 'memory': 3740076407808.0,
 'node:10.0.128.245': 1.0,
 'node:10.0.136.10': 1.0,
 'node:10.0.139.129': 1.0,
 'node:10.0.145.110': 1.0,
 'node:10.0.145.189': 1.0,
 'node:10.0.146.221': 1.0,
 'node:10.0.150.169': 1.0,
 'node:10.0.150.186': 1.0,
 'node:10.0.152.54': 1.0,
 'node:__internal_head__': 1.0,
 'object_store_memory': 3382286745600.0}


/opt/conda/envs/hft4/lib/python3.11/site-packages/dask/config.py:742: FutureWarning: Dask configuration key 'shuffle' has been deprecated; please use 'dataframe.shuffle.algorithm' instead
  warnings.warn(


In [3]:
# %%writefile cleaners.py

# from typing import *
# import pandas as pd
# from synthergent import Cleaner
# from pydantic import root_validator
# from synthergent.base.util import as_list, as_set



# class AddSuffix(Cleaner):
#     class Params(Cleaner.Params):
#         reps: int = 1
#         col: str = 'generations'
#         suffix: str = ' lol'
    
#     def clean(
#             self,
#             data: pd.DataFrame,
#             **kwargs,
#     ) -> pd.DataFrame:
#         for _ in range(self.params.reps):
#             data[self.params.col] = data[self.params.col].astype(str) + self.params.suffix
#         return data

# class FilterColumnsByValue(Cleaner):
#     class Params(Cleaner.Params):
#         exact_match: Dict[str, List] = {}
#         contains: Dict[str, List] = {}

#         @root_validator(pre=False)
#         def _check_params(cls, params: Dict) -> Dict:
#             if len(params['exact_match']) == len(params['contains']) == 0:
#                 raise ValueError('You must pass at least one parameter')
#             return params
    
#     def clean(
#             self,
#             data: pd.DataFrame,
#             **kwargs,
#     ) -> pd.DataFrame:
#         query_str = []

#         if len(self.params.exact_match) > 0:
#             exact_match_query_str = []
#             for col, values in self.params.exact_match.items():
#                 values = as_list(as_set(values))
#                 exact_match_query_str.append(f'({col} in {values})')
#             exact_match_query_str: str = '(' + ' and '.join(exact_match_query_str).strip() + ')'
#             query_str.append(exact_match_query_str.strip())

#         if len(self.params.contains) > 0:
#             contains_query_str = []
#             for col, values in self.params.contains.items():
#                 col_contains_query_str = []
#                 for value in as_list(as_set(values)):
#                     col_contains_query_str.append(f'{col}.str.contains("{value}", case=False, na=False)')
#                 col_contains_query_str: str = '(' + ' or '.join(col_contains_query_str).strip() + ')'
#                 contains_query_str.append(col_contains_query_str)
#             contains_query_str: str = '(' + ' and '.join(contains_query_str).strip() + ')'
#             query_str.append(contains_query_str.strip())
#         query_str: str = ' and '.join(query_str).strip()
#         if len(query_str) == 0:
#             raise ValueError('Created empty query string')
#         return data.query(query_str)


In [4]:
# import cleaners, importlib
# importlib.reload(cleaners)
# import cleaners

In [5]:
common_crawl_index_fm = FileMetadata.of(
    path='s3://commoncrawl/cc-index/table/cc-main/warc/crawl=CC-MAIN-2024-42/subset=warc/',
    format='parquet',
)
common_crawl_index_files: List[str] = common_crawl_index_fm.list(file_glob='*.parquet')
for i, x in enumerate(common_crawl_index_files):
    print(f'[{i:03}] {x}')

[000] s3://commoncrawl/cc-index/table/cc-main/warc/crawl=CC-MAIN-2024-42/subset=warc/part-00000-0c083cf2-c0ed-42ad-af5c-44f7548e96a0.c000.gz.parquet
[001] s3://commoncrawl/cc-index/table/cc-main/warc/crawl=CC-MAIN-2024-42/subset=warc/part-00001-0c083cf2-c0ed-42ad-af5c-44f7548e96a0.c000.gz.parquet
[002] s3://commoncrawl/cc-index/table/cc-main/warc/crawl=CC-MAIN-2024-42/subset=warc/part-00002-0c083cf2-c0ed-42ad-af5c-44f7548e96a0.c000.gz.parquet
[003] s3://commoncrawl/cc-index/table/cc-main/warc/crawl=CC-MAIN-2024-42/subset=warc/part-00003-0c083cf2-c0ed-42ad-af5c-44f7548e96a0.c000.gz.parquet
[004] s3://commoncrawl/cc-index/table/cc-main/warc/crawl=CC-MAIN-2024-42/subset=warc/part-00004-0c083cf2-c0ed-42ad-af5c-44f7548e96a0.c000.gz.parquet
[005] s3://commoncrawl/cc-index/table/cc-main/warc/crawl=CC-MAIN-2024-42/subset=warc/part-00005-0c083cf2-c0ed-42ad-af5c-44f7548e96a0.c000.gz.parquet
[006] s3://commoncrawl/cc-index/table/cc-main/warc/crawl=CC-MAIN-2024-42/subset=warc/part-00006-0c083cf2-c

In [6]:
with Timer():
    common_crawl_index_sample = pd.read_parquet(random.Random(42).choice(common_crawl_index_files))[
        ['url', 'content_languages']
    ]
    with pd_display() as display:
        display(common_crawl_index_sample.sample(n=10, random_state=1))

Started at 2024-12-15T17:54:31.174300+00:00...


,url,content_languages
1277709,https://www.edim.tv/tag/coffee-press/,"rus,eng"
3856450,https://forums.sonarr.tv/t/custom-regex-for-nzedb-indexers/4161,eng
1909841,https://www.hilltribe.tv/tag/video-recipe-agency-uk/,eng
1650179,https://www.funnycat.tv/video/every-single-video-of-the-land-of-boggs/sz-g5jMfKs4,eng
309051,https://aeolos.tv/tags/avgoustos/,ell
3413121,https://racemedia24.tv/2023/06/03/fia-bergrennen-ecce-homo-sternberk-cze/nggallery/page/2,"deu,eng"
4001842,https://superporno.tv/search/%D0%B2%D0%B0%D0%BB%D0%BE%D0%BC%D0%BF%D0%BE%D1%80%D0%BD%D0%BE/,"rus,eng"
7256111,https://www.idrlin.com.tw/%E6%8B%89%E7%9A%AE%E6%89%8B%E8%A1%93/%E6%8B%89%E7%9A%AE-%E8%87%89%E9%83%A8%E4%B8%8B%E5%9E%82%E6%95%91%E6%98%9F,"zho,eng"
8011274,https://www.mysheros.com.tw/SalePage/Index/10188513,"zho,eng"
545548,https://avtkhyber.tv/ilaqai-khabroona-mardan-17-07-2017/,eng


...completed in 32.08 seconds.


In [7]:
len(common_crawl_index_sample)

9000759

In [8]:
import synthergent.cleaner

coffee_press_articles = Synthergent.of(
    Cleaner.of(
        'StringCleaner',
        params=dict(
            col='url',
            cleaner=lambda url: urllib.parse.unquote(str(url)),
        )
    ),
    Cleaner.of(
        'FilterColumnsByValue', 
        params=dict(
            contains={
                'url': ['coffee press', 'coffee-press'],
            }
        )
    )
)

In [ ]:
with pd_display() as display:
    display(coffee_press_articles(data=common_crawl_index_sample))

Synthergent:   0%|          | 0/2 [00:00<?, ?step/s]

In [ ]:
with pd_display() as display:
    display(coffee_press_articles(
        data=common_crawl_index_sample,
        scaling=dict(
            parallelize='processes',
            max_workers=3,
            batch_size=1e6,
        )
    ))

In [11]:
common_crawl_index_fm_smaller = FileMetadata.of(
    path='s3://commoncrawl/cc-index/table/cc-main/warc/crawl=CC-MAIN-2024-42/subset=warc/',
    format='parquet',
    file_glob='part-0000*.parquet',
)

In [12]:
with Timer():
    common_crawl_index = Reader.of(
        'parquet',
        data_schema={
            'url': 'text', 
            'content_languages': 'text',
        },
    ).read(
        common_crawl_index_fm_smaller,
        read_as='dask',
    ).persist(wait=True)

Started at 2024-12-15T17:56:06.644640+00:00...
...completed in 26.73 seconds.


In [13]:
print(f'{len(common_crawl_index) / 1e6:.1f} million rows')

105.8 million rows


In [14]:
common_crawl_index.npartitions

76

In [15]:
with pd_display() as display:
    display(coffee_press_articles(
        data=common_crawl_index,
        scaling=dict(
            parallelize='ray',
        )
    ))

Synthergent:   0%|          | 0/2 [00:00<?, ?step/s]

,content_languages,url
816880,eng,https://buyscribblesdesigns.blogspot.com/2013/04/919-coffee-press-250.html
1087223,eng,https://blucentreelectronics.com/product/aeropress-original-coffee-press-3-in-1-brew-method-combines-french-press/
219138,eng,https://www.with-heart-and-hands.com/2012/12/making-coffee-press-wrap.html?showComment=1355132820763
246382,eng,https://withcues.com/products/bialetti-cold-brew-coffee-press
247114,eng,https://withcues.com/products/travel-coffee-press
1380744,eng,https://www.woodcockcycle.com/product/aerobie-aeropress-go-travel-coffee-press-41966.htm
344797,eng,https://home.woot.com/offers/aeropress-original-coffee-press-2?ref=w_cnt_odet_bs_1
282288,eng,https://workwithwire.com/fieldl/stanley-coffee-press-106027/
365800,tur,https://www.enplus.com.tr/bialetti-0002390nw-coffee-press-preziosa-600-ml-19431
1211154,tur,https://www.starbucks.com.tr/menu/product/coffee-press-P194
